# Overlay per image ID

Introdueix un `image_id` (nom del fitxer, per exemple `16B0001851_Block_Region_1_5_7_xini_13971_yini_64906.jpg`) i aquest notebook mostrarà:

- imatge de teixit (H&E)
- màscara GT en mode overlay transparent

Aquest notebook usa els intervals de `src/training_conchv2.py`:
- `25-74 -> GG3`
- `75-174 -> GG4`
- `175-255 -> GG5`
- resta -> `NC`

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

# Detecta l'arrel del projecte (SICAPv2)
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "images").exists():
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "images").exists() and (parent / "masks").exists():
            REPO_ROOT = parent
            break

IMAGES_DIR = REPO_ROOT / "images"
MASKS_DIR = REPO_ROOT / "masks"

CLASS_NAMES = ["NC", "GG3", "GG4", "GG5"]
CLASS_COLORS = {
    0: (50, 50, 50),
    1: (46, 204, 113),
    2: (241, 196, 15),
    3: (231, 76, 60),
}

# Intervals de src/training_conchv2.py
MASK_LUT = np.zeros(256, dtype=np.int64)
MASK_LUT[25:75] = 1
MASK_LUT[75:175] = 2
MASK_LUT[175:] = 3


def read_rgb(path: Path) -> np.ndarray:
    buf = np.fromfile(str(path), dtype=np.uint8)
    bgr = cv2.imdecode(buf, cv2.IMREAD_COLOR)
    if bgr is None:
        raise FileNotFoundError(path)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def read_mapped_mask(path: Path) -> np.ndarray:
    buf = np.fromfile(str(path), dtype=np.uint8)
    raw = cv2.imdecode(buf, cv2.IMREAD_GRAYSCALE)
    if raw is None:
        raise FileNotFoundError(path)
    return MASK_LUT[raw]


def colorize_mask(mapped: np.ndarray) -> np.ndarray:
    rgb = np.zeros((*mapped.shape, 3), dtype=np.uint8)
    for c, col in CLASS_COLORS.items():
        rgb[mapped == c] = col
    return rgb


def make_overlay(rgb: np.ndarray, mapped: np.ndarray, alpha: float = 0.55) -> np.ndarray:
    base = rgb.astype(np.float32) / 255.0
    colored = colorize_mask(mapped).astype(np.float32) / 255.0
    out = base.copy()
    tissue = mapped != 0  # NC sense overlay
    out[tissue] = (1 - alpha) * base[tissue] + alpha * colored[tissue]
    return np.clip(out, 0.0, 1.0)


def class_summary(mapped: np.ndarray) -> str:
    bc = np.bincount(mapped.ravel().astype(np.int64), minlength=4)
    parts = [f"{CLASS_NAMES[i]} {100 * bc[i] / mapped.size:.1f}%" for i in range(4) if bc[i] > 0]
    return ", ".join(parts)

In [ ]:
# Escriu aquí l'image_id (fitxer dins images/ i masks/)
image_id = "16B0001851_Block_Region_1_5_7_xini_13971_yini_64906.jpg"

img_path = IMAGES_DIR / image_id
mask_path = MASKS_DIR / image_id

if not img_path.exists():
    raise FileNotFoundError(f"No existeix la imatge: {img_path}")
if not mask_path.exists():
    raise FileNotFoundError(f"No existeix la màscara: {mask_path}")

rgb = read_rgb(img_path)
mapped = read_mapped_mask(mask_path)
over = make_overlay(rgb, mapped, alpha=0.55)

fig, axs = plt.subplots(1, 2, figsize=(12, 5))
axs[0].imshow(rgb)
axs[0].set_title("H&E")
axs[0].axis("off")

axs[1].imshow(over)
axs[1].set_title(f"Overlay GT\n{class_summary(mapped)}")
axs[1].axis("off")

plt.tight_layout()
plt.show()